In [2]:
# import essential
import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")

if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))



In [ ]:
# run in parallel

import multiprocessing as mp
from pathlib import Path

from general_utils import find_ephys_sessions
from monotonic_mp import run_all_sessions_parallel

# IMPORTANT in notebooks
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass

PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")
OUT_DIR = Path("/root/capsule/scratch/monotonic_outputs_n_4")

LATENT_NAMES = [
    "ForagingCompareThreshold-value-1",
    "QLearning_L2F1_softmax-deltaQ-1",
    "QLearning_L2F1_softmax-sumQ-1",
]

ACTIVITY_WINDOWS = [(-1, 0), (0.2, 2),(-3,0)]

_, _, sessions = find_ephys_sessions()


_results = run_all_sessions_parallel(
    list(sessions),
    psth_dir=PSTH_DIR,
    results_dir=RESULTS_DIR,
    out_dir=OUT_DIR,
    latent_names=LATENT_NAMES,
    activity_windows=ACTIVITY_WINDOWS,
    max_workers=4,   # scale up later if stable
)


failed = [r for r in _results if not r.get("ok", False)]

print(f"Failed sessions: {len(failed)}")

failed_sessions = [r.get("session") for r in failed]
print(failed_sessions[:20])

sessions=failed_sessions
_results = run_all_sessions_parallel(
    list(sessions),
    psth_dir=PSTH_DIR,
    results_dir=RESULTS_DIR,
    out_dir=OUT_DIR,
    latent_names=LATENT_NAMES,
    activity_windows=ACTIVITY_WINDOWS,
    max_workers=4,   # scale up later if stable
)

In [3]:
# Load summary (as you already do)
# -----------------------------
import importlib
import behavior_qc_visualization

importlib.reload(behavior_qc_visualization)


from behavior_qc_visualization import load_behavior_model_summary_csv
from behavior_qc_metrics_summary import append_model_criteria_result

summary = load_behavior_model_summary_csv(
    "/root/capsule/scratch/behavior_model_summary_ephys_sessions.csv"
    # "/root/capsule/scratch/behavior_model_summary_general_behavior.csv"
)
summary = append_model_criteria_result(summary)


# -----------------------------
# Load combined_df + summarize (UPDATED: add summary + criteria_col)
# -----------------------------
from check_monotonic import load_and_combine_monotonic_unit_dfs
from check_monotonic import summarize_significant_and_monotonic_fractions

combined_df = load_and_combine_monotonic_unit_dfs(
    "/root/capsule/scratch/monotonic_outputs_n_4/ForagingCompareThreshold-value-1/win_m1__0",
    pattern="*.csv",
    recursive=True,
    add_source_file=True,
)
print("combined_df shape (before filtering inside function):", combined_df.shape)

out = summarize_significant_and_monotonic_fractions(
    combined_df,
    alpha=0.05,
    p_col="spearman_p",
    monotonic_col="is_monotonic_gt_thr",
    include_overall=True,
    summary=summary,  # NEW
    # Default criteria_col is 'QLearning_L1F1_CK1_softmax_pass_all_criteria'
    # Override here if you want a different model's criteria:
    # criteria_col="QLearning_L2F1_softmax_pass_all_criteria",
)

# Optional: see how many rows survived the session filter
if out.get("session_filter_applied", False):
    print(
        "Session filter applied:",
        out["criteria_col"],
        "| rows:",
        out["n_rows_before_session_filter"],
        "->",
        out["n_rows_after_session_filter"],
        "| n_sessions_passing:",
        out["n_sessions_passing_criteria"],
    )

overall = out["overall"]
print("Fraction significant (among valid p):", overall["frac_significant_among_valid_p"])
print("Fraction monotonic among significant:", overall["frac_monotonic_among_significant"])
print("n_sig:", overall["n_significant"], "n_sig_monot:", overall["n_significant_and_monotonic"])

sig_df = overall["significant_df"]
sig_monot_df = overall["significant_monotonic_df"]

for label, res in out["by_brain_region_group"].items():
    print(
        label,
        "n_total:", res["n_total_rows"],
        "frac_sig:", res["frac_significant_among_valid_p"],
        "frac_monot|sig:", res["frac_monotonic_among_significant"],
        "n_sig:", res["n_significant"],
        "n_sig_monot:", res["n_significant_and_monotonic"],
    )


combined_df shape (before filtering inside function): (25055, 80)
Session filter applied: QLearning_L1F1_CK1_softmax_pass_all_criteria | rows: 25055 -> 16655 | n_sessions_passing: 26
Fraction significant (among valid p): 0.40246542393265183
Fraction monotonic among significant: 0.41296877334528614
n_sig: 6693 n_sig_monot: 2764
[MD] n_total: 2223 frac_sig: 0.40018026137899954 frac_monot|sig: 0.4369369369369369 n_sig: 888 n_sig_monot: 388
[SI,MA] n_total: 462 frac_sig: 0.43383947939262474 frac_monot|sig: 0.4 n_sig: 200 n_sig_monot: 80
[PL5,PL6a,ILA5,ILA6a] n_total: 295 frac_sig: 0.4542372881355932 frac_monot|sig: 0.5074626865671642 n_sig: 134 n_sig_monot: 68
[MOs2/3,MOs5,MOs6a] n_total: 787 frac_sig: 0.49363867684478374 frac_monot|sig: 0.538659793814433 n_sig: 388 n_sig_monot: 209
[ORBm1,ORBm5,ORBm2/3,ORBm6a] n_total: 247 frac_sig: 0.3805668016194332 frac_monot|sig: 0.425531914893617 n_sig: 94 n_sig_monot: 40


In [4]:
out['by_brain_region_group'].keys()

dict_keys(['[MD]', '[SI,MA]', '[PL5,PL6a,ILA5,ILA6a]', '[MOs2/3,MOs5,MOs6a]', '[ORBm1,ORBm5,ORBm2/3,ORBm6a]'])

In [8]:
out['by_brain_region_group']['[SI,MA]' ]

{'alpha': 0.05,
 'p_col': 'spearman_p',
 'monotonic_col': 'is_monotonic_gt_thr',
 'n_total_rows': 462,
 'n_rows_with_valid_p': 461,
 'n_significant': 200,
 'frac_significant_among_valid_p': 0.43383947939262474,
 'n_significant_with_valid_monotonic': 200,
 'n_significant_and_monotonic': 80,
 'frac_monotonic_among_significant': 0.4,
 'significant_df':        unit_index                                       session_name  \
 10467         288  ecephys_769884_2025-01-16_18-33-11_sorted_2025...   
 10470         191  ecephys_769884_2025-01-16_18-33-11_sorted_2025...   
 10536         445  ecephys_769884_2025-01-16_18-33-11_sorted_2025...   
 10540         454  ecephys_769884_2025-01-16_18-33-11_sorted_2025...   
 10555         197  ecephys_769884_2025-01-16_18-33-11_sorted_2025...   
 ...           ...                                                ...   
 22541        1840  ecephys_781471_2025-03-31_14-33-03_sorted_2025...   
 22597        1828  ecephys_781471_2025-03-31_14-33-03_sorted_202

In [10]:
row = combined_df[
    (combined_df['unit_index'] == 365) &
    (combined_df['brain_region'].isin(['PL5','PL6a','ILA5','ILA6a']))
]

row


,unit_index,session_name,latent_name,activity_min_threshold,min_average_firing_rate,avg_firing_rate_window,calculation_window_start,calculation_window_end,n_trials_used,n_trials_used_gt_thr,...,q3_mean_activity,q3_trial_mean_activity_list,q3_sem_activity,q3_n_gt_thr,q3_mean_activity_gt_thr,q3_trial_mean_activity_list_gt_thr,q3_sem_activity_gt_thr,brain_region,ccf_location,source_file
9824,365,ecephys_769884_2025-01-15_16-12-59_sorted_2025...,ForagingCompareThreshold-value-1,0.0,0.0,13.733739,-4.0,3.0,470,470,...,16.754237,"[2.0, 1.0, 2.0, 0.0, 4.0, 13.0, 8.0, 17.0, 12....",1.130929,118,16.754237,"[2.0, 1.0, 2.0, 0.0, 4.0, 13.0, 8.0, 17.0, 12....",1.130929,PL6a,"{'best_electrode': 73, 'shank': 1, 'probe': 'P...",ecephys_769884_2025-01-15_16-12-59_sorted_2025...


In [ ]:
# Example: randomly open 8 significant & monotonic neurons

import importlib
import check_monotonic

# Reload the entire module
importlib.reload(check_monotonic)

from check_monotonic import show_unit_figures_from_df_inline

opened_paths = show_unit_figures_from_df_inline(
    out['by_brain_region_group']['[SI,MA]' ]['significant_monotonic_df'],   # your filtered DataFrame
    root="/root/capsule/scratch/raster_plot",
    session_col="session_name",
    latent_col="latent_name",
    unit_col="unit_index",
    n=100,                        # randomly open 8 figures
    random_state=41,            # reproducible sampling
)

print(f"Opened {len(opened_paths)} figures.")


In [ ]:
# plot single neuron activity

import importlib
import plot_raster

importlib.reload(plot_raster)

from plot_raster import plot_raster_and_quantile_psth_by_latent



from pathlib import Path
import numpy as np

from general_utils import find_ephys_sessions, smart_read_csv
from behavior_utils import find_trials, get_fitted_model_names
from nwb_utils import NWBUtils
from create_psth import load_zarr
from plot_raster import plot_raster_and_quantile_psth_by_latent


PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")
OUTDIR = Path("/root/capsule/scratch/raster_plot")

LATENTS = [
    "QLearning_L2F1_softmax-deltaQ-1",
    "QLearning_L2F1_softmax-reward",
    "QLearning_L2F1_softmax-sumQ-1",
]
latent_names = ['deltaQ-1','reward','sumQ-1']

ALIGN_TO = "go_cue"
TIME_WIN = (-3, 4)


OUTDIR.mkdir(parents=True, exist_ok=True)
_, _, sessions = find_ephys_sessions()
print("Sessions:", sessions)

sessions=["ecephys_769884_2025-01-15_16-12-59_sorted_2025-04-24_17-11-57"]




for s in sessions:
    print(f"\n[{s}] models:", get_fitted_model_names(session_name=s))
    nwb, _ = NWBUtils.combine_nwb(session_name=s)
    psth = load_zarr(str(PSTH_DIR / f"{s}_0.2s.zarr"))
    beh = smart_read_csv(str(RESULTS_DIR / f"behavior_summary-{s}.csv"))
    trials = np.asarray(find_trials(nwb, "response"))
    save_dir = OUTDIR / s
    save_dir.mkdir(exist_ok=True)

    i=0
    for col in LATENTS:
        if col not in beh.columns:
            print(f"  skip: {col} (missing)")
            continue
        vals = beh[col][0]

        plot_raster_and_quantile_psth_by_latent(
            source=psth,
            latent_values=vals,              # full vector
            latent_trial_ids=trials,
            unit_ids=[365],
            align_to_event=ALIGN_TO,
            time_window=TIME_WIN,
            n_bins=4,
            binning="equal",
            quantile_stat="mean",
            ci="sem",
            title_prefix=None,
            figsize=(6, 5),
            save_path="/root/capsule/scratch/tmp",
            cmap_name="coolwarm",
            raster_colormap=False,
            show_colormap=True,
            save_prefix=f'{col}_',
            latent_name=latent_names[i],
            show=True,
            min_trial_rate=0,
            sort_order='not_sort',
            dpi=600
        )
        i=i+1
        print(f"  plotted: {col}")




